# Class 11 - RNAseq Analysis III
Downstream analysis of DEGs, clustering and pathway analysis

Start by Installing and Importing the necessary packages. Note that we will need to set some specific settings to get everything to run properly in Colaboratory.




In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.stats.multitest as smm
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import seaborn as sns
import csv
import os
import sys
import argparse
from time import time
from pprint import pprint

Setting up some other Pandas and python presets

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import gseapy as gp
from gseapy import Msigdb
import GEOparse

In [ ]:
import ftplib
import os
import urllib.parse
import gzip

In [ ]:
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)

# We're also going to tell Jupyter to use inline plotting instead of notebook plotting
# It basically means you don't have to use plt.show() in every cell
%matplotlib inline

# and this command will allow multiple outputs from the same cell, rather than just the last one run
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Enrichment analysis

Besides looking at each DEG one-by-one, we can ask what shared biological pathways or processes do these DEGs come from. By looking at the overlap of our DEG list with predefined annotated lists of genes associated with specific biological functions, and identifying the 'enriched' processes as those that are disproportionately present in our DEGs, we can gain insight into what might be happening in our experiment.

First, let's look at a statistical approach we can use to test for enrichment.

#### **The Hypergeometric Test**

Interpretation:

 - H0: the number of successes, given the number of attempts, occured by random chance.
 - H1: the number of successes is over-represented or under-represented, given the number of attempts.

Assumptions:

 - Each attempt represents an independent event.
 - Each attempt represents drawing from a population, without replacement (i.e. the universe of possibilities gets smaller with each draw).


An abstract example:
Suppose we have a collection of 20 marbles, of which 7 are blue (the rest are red). To know the probability of finding a given number of blue marbles (e.g. = 1 or fewer) if we choose at random 12 of the 20 marbles, we perform the following calculation:

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

In [ ]:
TotalPop = 20 # size of total population
NumSuccess = 7 # number of successes in the total population
NumTries = 12 # number of tries
SuccessfulTriesorFewer = 1 # Max number of successes among the tries

# calculates probability of finding x or fewer marbles
pfew = stats.hypergeom.cdf(SuccessfulTriesorFewer, TotalPop, NumSuccess, NumTries)
print(f"Probability of finding x or fewer marbles: {pfew:1.4f}")

To calculate the probability of finding x or more successes,


use `1 - stats.hypergeom.cdf()`:

In [ ]:
SuccessfulTriesorMore = 5 # number of successes among the tries
# calculates the probability of finding x or more
pmore = 1 - stats.hypergeom.cdf(SuccessfulTriesorMore,TotalPop,NumSuccess,NumTries)

print(f"Probability of finding x or more marbles: {pmore:1.4f}")

### Using Hypergeometric testing for assessing enrichment of specific pathways

A common application of hypergeometric testing is to assess the likelihood that genes perturbed under different conditions are associated with specific previously annotated functions or pathways.

We'll discuss more online resources that annotate pathways and networks during our Networks section of the class (week 8) but for now, the Molecular Signatures Database ([MSigDB](https://www.gsea-msigdb.org/gsea/msigdb/)) is one good resource for many gene sets of interest.

We're going to focus on looking at a couple of pathways from the "Hallmark Signatures" collection.

In [ ]:
HallmarkGenesets = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/HallmarkGeneSets_mSigDB.csv')
display(HallmarkGenesets)

For this first example, we're going to focus on TGFβ signaling:

In [ ]:
display(HallmarkGenesets['HALLMARK_TGF_BETA_SIGNALING'].dropna()) # using the .dropna() method to get rid of any NaN rows 

Our goal is to test the extent to which the top 100 differentially expressed genes are significantly enriched for genes in the TGFβ signaling pathway.

Recall that we need 4 key values to perform this calculation:

* Total population
* Number of successes in the population
* Size of selection
* Number of successes in the selection

The total population is the number of genes we tested with DESeq2:

Let's load the DEG results



In [ ]:
deseq_melanoma_results = pd.read_csv("~/LECTURE_MATERIALS/DataFiles/deseq_results_melanoma.csv",index_col=0)
display(deseq_melanoma_results)

In [ ]:
# Filter to get only significant results, and sort by fold change
sig_results = deseq_melanoma_results[(deseq_melanoma_results['padj'] < 0.05) &
			(abs(deseq_melanoma_results['log2FoldChange']) > 1)].sort_values('log2FoldChange', ascending=False)
display(sig_results)

Let's examine the overlap between the top 100 upregulated genes and the `HALLMARK_KRAS_SIGNALING_UP` signaling pathway (i.e. the genes that are upregulated by KRAS activation, as defined by [MSigDB](https://www.gsea-msigdb.org/gsea/msigdb/cards/HALLMARK_KRAS_SIGNALING_UP)).

First, we need to slice the top 100 upregulated genes from the DESeq results data frame

In [ ]:
#Pick the top 100 most increased genes from the top of the list
top_genes100 = sig_results.head(100).index.tolist()

print(top_genes100[:10])

Then, we will extract the genes associated with the `HALLMARK_KRAS_SIGNALING_UP` pathway

In [ ]:
# slice out gene names from Hallmark KRAS UP signaling:
pathway_genes = HallmarkGenesets.loc[:,'HALLMARK_KRAS_SIGNALING_UP'].dropna().tolist()
print(pathway_genes[:10])

Now, we can formulate the hypergeometric test

In [ ]:
#Population size = total number of genes tested
n_pop = len(deseq_melanoma_results.index)
print("Population:", n_pop)


#Successes in population = total pathway genes in population
num_successes = len(set(pathway_genes) & set(deseq_melanoma_results.index))
print("Successes in population: ", num_successes)

#Number of tries = the number of top 100 genes (i.e. 100)
num_tries = len(top_genes100)
print("Size of selection: ", num_tries)

# perform a set intersection to determine successes in tries
shared_genes = list(set(top_genes100) & set(pathway_genes))
print("Successes in selection:", len(shared_genes))

#Hypergeometric test
pmore = 1 - stats.hypergeom.cdf(len(shared_genes),n_pop,num_successes,num_tries)
print(f"P(overlap): {pmore:1.6f}")

Given that the p-value is << 0.05, we can conclude that the KRAS UP signaling pathway is significantly enriched in our top 100 most increased DEGs.


### <font color=brown>Hands on practice</font>

1. Perform a similar enrichment analysis for the `HALLMARK_HYPOXIA` gene set. Is this gene set significantly enriched among the top 100 variably expressed genes?


2. Try all 50 gene sets: which are significantly overlapping with the top DEGs? Which is the most significant?

In [ ]:
HallmarkGenesets.columns

## Gene set enrichment analysis (GSEA) with GSEAPY

The GSEAPY package provides functions to perform GSEA with pre-defined gene sets. In contrast to hypergeometric enrichment, GSEA works using the a ranked list of all genes. The basic idea for GSEA enrichment is that genes from a non-enriched gene set will be randomly distributed within this ranked list, while interesting gene sets will be disproportionately located towards the top or bottom of the ranked list. GSEA reports results are Normalized Enrichment Scores (NES) and p-values.

A standard workflow is to rank your genes by fold change (most increased to most decreased) and use GSEA to test for gene sets preferentially located towards. the top (increased) or bottom (decreased) of your ranked gene list

In [ ]:
msig = Msigdb()
msig.list_dbver()

In [ ]:
#First we will use GSEAPY to retrieve the human msigdb hallmark gene sets
gsets = msig.get_gmt(category='h.all', dbver="2026.1.Hs")
print(gsets.keys())



In [ ]:
gsets['HALLMARK_KRAS_SIGNALING_UP']

In [ ]:
print(deseq_melanoma_results.head())

In [ ]:
sorted_deseq_melanoma_results = deseq_melanoma_results.sort_values('log2FoldChange', ascending=False)
print(sorted_deseq_melanoma_results["log2FoldChange"].head())

In [ ]:
#Run GSEA on our DESeq2 results
#Set the index column to pass to the ranking function
gsea_res = gp.prerank(rnk = sorted_deseq_melanoma_results["log2FoldChange"],
                      gene_sets=gsets,
                      outdir=None)

In [ ]:
#Let's take a look at the most enriched GSEA results
display(gsea_res.res2d.sort_values('NES', ascending=False))


We can visualize how GSEA calculates its enrichment scores. For each pathway to test, GSEA draws a line that starts at 0 at the highest ranked gene and moves right across the genes. It slightly increases whenever it encounters a pathway gene and decreases whenever it encounters a non-pathway gene. These changes are calibrated so that a pathway randomly distributed over the ranking will have the line as close to 0 as possible, while enriched pathways will deviate from 0 towards the start (increased) or end (decreased) of the line. This area under the curve is the enrichment score.

A visual example might be easiest!

In [ ]:
ranked_genesets = gsea_res.res2d['Term'].tolist()
print(ranked_genesets)

In [ ]:
ranked_genesets[0]

In [ ]:
gsea_res.plot(terms=ranked_genesets[1])

In [ ]:
gsea_res.plot(terms=ranked_genesets[-1])

In [ ]:
gp.dotplot(gsea_res.res2d,
             column="FDR q-val",
             title='Melanoma: Primary melanocytes vs metastatic',
             size=6, # adjust dot size
             figsize=(4,5), cutoff=0.25, show_ring=False)

### Functional enrichment analysis

Often when we assess enrichment of functionally related gene sets, we would like to investigate relevant related sets of genes form multiple sources. Customizing this analysis is possible using Python by extending the strategies laid out above; however, there are multiple online tools that also help to make this process more straightforward.

Some useful web platforms for enabling enrichment analysis of related gene sets are:
* [ShinyGO](https://bioinformatics.sdstate.edu/go/)
* [Enrichr](https://maayanlab.cloud/Enrichr/enrich)
* [g:Profiler](https://biit.cs.ut.ee/gprofiler/gost)

To illustrate the utility of these websites, let's copy the top 100 variably expressed genes from the spreadsheet that we created earlier, and use it perform functional enrichment analysis.

In [ ]:
print("\n".join(top_genes100))

# Getting publicly available datasets to do your own analysis
*tutorial courtesy Fred Mast*

RNAseq (and microarray) datasets for millions of samples from hundreds of thousands of studies are available at NCBI GEO (Gene Expression Omnibus). This database includes transcriptome data from multiple species (including host, pathogen, cell lines), experimental designs, treatments, etc.

The basic organizing principle is based on GEO DataSets (GSEXXXXX), comprising multiple GEO samples (GSMXXXXX).

Once you know the datasets or samples you are interested in, we can use Python to directly download and work with the data.

A useful package to help with importing data from GEO is `GEOparse`.

In [ ]:
# Retrieve the GEO dataset with accession number GSE88741
# This contains the melanoma data we have been using for this course so far...
gse = GEOparse.get_GEO(geo="GSE88741")

### Exploring Metadata of GEO Dataset GSE88741

Once we have the GEO dataset GSE88741 loaded, it's beneficial to examine the metadata. Metadata in the context of GEO datasets provides comprehensive details about the study, which are crucial for understanding the experimental setup and for subsequent data analysis.

Reviewing the metadata is essential because it:
- Provides context about the data, including the biological species studied, the experimental conditions, and the data collection methods.
- Helps in verifying the relevance of the dataset to specific research questions.
- Ensures the integrity and reproducibility of scientific analysis by offering detailed experimental protocols.

By familiarizing yourself with the dataset's metadata, you ensure a thorough understanding of the data's origins and characteristics, which is crucial for conducting accurate and effective analyses.


Here's how we can extract and view this metadata using Python:

In [ ]:
display(gse.metadata)

   The line above accesses the `metadata` attribute of the `gse` object, which stores all the metadata as a dictionary. Each entry in this dictionary corresponds to different metadata fields, such as the study's title, contributors, and publication information.

   You can access each of the elements in the dictionary using the syntax below.

In [ ]:
#extracting data from the individual fields
gse.metadata['sample_id']

You can also print out the phenotype description with the `phenotype_data` attribute.

#### Overview of `gse.phenotype_data`
This command returns a DataFrame containing detailed annotations for each sample in a GEO Series (GSE). Understanding the structure of this DataFrame will help you effectively utilize the data:

- **Rows**: Each row represents a distinct sample, identified as a GSM object within the GEO repository.
- **Columns**: The columns provide various details about each sample. The available information can vary by dataset but typically includes:

#### Detailed Breakdown of Phenotype Data Columns
Here’s a brief explanation of some key columns you may encounter in the phenotype data DataFrame:

- **`title`**: Descriptive title of the sample.
- **`geo_accession`**: Unique identifier for the sample within the GEO database.
- **`status`**: Current status of the sample's public availability.
- **`submission_date``, `last_update_date`**: Dates relevant to the sample's submission and last update on GEO.
- **`type`**: Type of biological material or experiment (e.g., RNA, DNA).
- **`source_name_ch1`**: The biological source of the sample (e.g., specific tissue or cell type).
- **`organism_ch1`**: The organism from which the sample was derived.
- **`taxid_ch1`**: Taxonomic identifier for the organism.
- **`characteristics_ch1.0.cell type`**, `characteristics_ch1.1.Stage`**: Specific characteristics such as cell type or developmental stage.
- **`treatment_protocol_ch1`**, `growth_protocol_ch1`**: Details on the treatment or growth conditions applied to the sample.
- **`molecule_ch1`**: Type of molecule analyzed (e.g., total RNA, genomic DNA).
- **`extract_protocol_ch1`**: Protocol used to extract the biological material.
- **`data_processing`**: Methods used for processing the data.
- **`platform_id`**: Identifier for the platform used for data collection.
- **`contact_name`**, `contact_email`**: Contact information of the primary researcher.
- **`contact_laboratory`**, `contact_department`**, `contact_institute`**: Laboratory and institutional affiliations.
- **`contact_address`**, `contact_city`**, `contact_state`**, `contact_zip/postal_code`**, `contact_country`**: Full contact address.
- **`instrument_model`**: The model of the instrument used for data collection.
- **`library_selection`**, `library_source`**, `library_strategy`**: Library preparation details.
- **`relation`**: Links to related databases or resources (e.g., SRA, BioSample).
- **`supplementary_file_1`**: URL or reference to supplementary files.
- **`series_id`**: Identifier for the series to which the sample belongs.
- **`data_row_count`**: Number of data rows specific to the sample.

#### Utilizing Phenotype Data
Phenotype data provides a comprehensive view of each sample, enabling researchers to:
- Understand experimental variables and their impact on gene expression.
- Identify samples for comparative studies based on specific characteristics.
- Ensure the reproducibility of experiments by providing detailed protocols and conditions.


In [ ]:
#Phenotype data is a table of sample-specific data
display(gse.phenotype_data)

### Automating the Download of Raw Counts Data from GEO Using Python

When working with GEO datasets, a common requirement is to download and analyze the raw counts or FPKM data associated with the samples. Python, with libraries like `GEOparse`, provides a flexible and powerful way to automate this process by leveraging metadata from the dataset. Here, we discuss a streamlined process to automatically download the FPKM data for further analysis, using metadata to dynamically retrieve file URLs.

#### Overview of the Process
The process involves several steps:
1. **Load the GEO Dataset**: Initially, we load the dataset using `GEOparse`, which provides access to all related metadata and data links.
2. **Examine Metadata**: The metadata contains crucial information such as the URL of supplementary files which often include raw counts or FPKM data.
3. **Download and Decompress the File**: We then use this URL to download and decompress the file, preparing it for analysis.
4. **Load Data for Analysis**: Finally, the data is loaded into a DataFrame for subsequent analysis.

#### Detailed Explanation of Functions

1. **Function to Download and Decompress the Supplementary File**
   - **Purpose**: Handles the downloading and decompression of files from FTP servers specified in the GEO dataset's metadata.
   - **Process**:
     - **Parse URL**: Extract the hostname and path from the FTP URL using `urllib.parse.urlparse`.
     - **FTP Connection**: Connect to the FTP server using `ftplib.FTP` and navigate to the directory containing the file.
     - **Download**: Retrieve the `.gz` file and save it locally.
     - **Decompress**: Open the downloaded gzip file, extract its contents, and save them to a specified output path.
     - **Cleanup**: Remove the original compressed file to conserve disk space.

   ```python
   def download_and_decompress_ftp(ftp_url, output_path):
       ...
   ```

In [ ]:
# Function to download and decompress the supplementary file
def download_and_decompress_ftp(ftp_url, output_path):
    parse = urllib.parse.urlparse(ftp_url)
    ftp_host = parse.hostname
    ftp_path = parse.path

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Log in anonymously
        ftp.cwd(os.path.dirname(ftp_path))

        local_gz_path = output_path + '.gz'
        with open(local_gz_path, 'wb') as local_file:
            ftp.retrbinary('RETR ' + os.path.basename(ftp_path), local_file.write)
        print(f"Downloaded the file to {local_gz_path}")

        with gzip.open(local_gz_path, 'rb') as gzipped_file:
            with open(output_path, 'wb') as decompressed_file:
                decompressed_file.write(gzipped_file.read())
        print(f"Decompressed the file to {output_path}")

        os.remove(local_gz_path)

2. **Function to Automatically Retrieve FTP URL and Output Path from Metadata**
   - **Purpose**: Streamlines the retrieval of the supplementary file URL from the dataset's metadata and prepares the output path based on the file name.
   - **Process**:
     - **Retrieve URL**: Access the `supplementary_file` field in the metadata to get the URL.
     - **Prepare Output Path**: Construct the output path using the file name derived from the URL, ensuring it is decompressed correctly.
     - **Call Download Function**: Use the previously defined function to handle the download and decompression.

   ```python
   def prepare_download(gse, output_dir):
       ...
   ```

In [ ]:
# Function to automatically get the FTP URL and output path from metadata
def prepare_download(gse, output_dir):
    supplementary_file_url = gse.metadata.get('supplementary_file', [None])[0]
    if supplementary_file_url:
        file_name = os.path.basename(urllib.parse.urlparse(supplementary_file_url).path)
        output_path = os.path.join(output_dir, file_name.replace('.gz', ''))
        download_and_decompress_ftp(supplementary_file_url, output_path)
        return output_path
    else:
        print("No supplementary file URL found in metadata.")
        return None

#### Implementing the Download

```python
# Assuming 'gse' has already been loaded with GEOparse.get_GEO(geo="GSE88741")
output_directory = "/path/to/Downloads"
file_path = prepare_download(gse, output_directory)
```

#### Loading and Analyzing the Data

Once the file is downloaded and decompressed:
```python
if file_path and os.path.exists(file_path):
    fpkm_data = pd.read_csv(file_path, sep='\t', index_col=0)
    print(fpkm_data.head())
```

#### Adapting the Code for Other Datasets

- **Change the GEO Accession Number**: Replace `"GSE88741"` with any other valid accession number to download different datasets.
- **Modify the Output Directory**: Adjust the `output_directory` path based on your local or server file system environment.

In [ ]:
# Load a GEO dataset dynamically (already loaded, but you can adapt this code for other datasets)
# gse = GEOparse.get_GEO(geo="GSE88741")

# Specify the directory where files should be saved
output_directory = "."

# Automatically prepare the download
file_path = prepare_download(gse, output_directory)

# Load the file into a DataFrame if it exists
if file_path and os.path.exists(file_path):
    fpkm_data = pd.read_csv(file_path, sep='\t', index_col=0)
    print(fpkm_data.head())

In [ ]:
display(fpkm_data)

### Comparing the Use of Supplied FPKM Values vs. Processing Raw Data in GEO Datasets

When analyzing gene expression data from GEO datasets, researchers have two primary options: using pre-computed FPKM (Fragments Per Kilobase of transcript per Million mapped reads) values supplied with the dataset or processing the raw data themselves. Each approach has its own set of benefits and trade-offs.

#### **Using Supplied FPKM Values**

**Benefits:**
- **Time-Efficient:** Utilizing pre-computed FPKM values saves significant time and computational resources as the data is ready for immediate analysis.
- **Ease of Use:** These values are straightforward to use, making them accessible for researchers without extensive computational backgrounds.
- **Standardization:** FPKM normalization facilitates comparisons of gene expression across samples within the dataset, as it adjusts for gene length and sequencing depth.

**Trade-offs:**
- **Less Control:** Researchers have less control over the normalization process and cannot account for batch effects or other confounding variables not considered in the original processing.
- **Generic Processing:** The one-size-fits-all approach in preprocessing may not be optimal for all types of analyses.
- **Potential for Bias:** Depending on how the FPKM values were computed, there might be biases that could affect the analysis, especially if the normalization assumptions are not suitable for the specific dataset.

#### **Processing Raw Data Yourself**

**Benefits:**
- **Customization:** Allows researchers to tailor data processing workflows to fit specific research needs, optimizing the handling of confounders and technical variations.
- **Advanced Analysis:** Enables the use of sophisticated statistical models and the latest data processing algorithms, potentially leading to more robust and accurate results.
- **Transparency and Reproducibility:** Processing raw data enhances the transparency of the research process and allows for adjustments and improvements in data handling, promoting reproducibility.

**Trade-offs:**
- **Resource Intensive:** Requires more computational resources and time, especially for large datasets.
- **Technical Expertise Required:** Demands a higher level of bioinformatics knowledge to handle raw data appropriately and to choose and configure the right tools.
- **Complexity:** Managing the entire data processing pipeline adds complexity to the research process, which can introduce errors if not done meticulously.

#### **Conclusion**

Choosing between using supplied FPKM values and processing raw data yourself depends largely on the specific needs of the research, the available resources, and the level of control and customization required. For exploratory studies where time and resources are limited, using pre-computed FPKM values may be beneficial. However, for in-depth studies that aim to uncover subtle biological mechanisms, processing raw data may provide more accurate and tailored insights.

## Obtaining standardised RNAseq counts with ARCHS4PY
Data uploaded to GEO can be normalized or processed in many different ways, depending on how the original authors analyzed the data. However, to maintain comparability, we would ideally like our RNAseq data to have been aligned and quantified as comparably as possible!

The ARCHS4 project is a regularly-updated effort from the Ma'ayan lab at Mount Sinai to make standardised RNAseq gene counts tables for human and mouse expression data easily available.

[The ARCHS4 project](https://https://maayanlab.cloud/archs4/)

It comes with a very useful python package: archs4py


In [ ]:
import archs4py

However, working with archs4py requires downloading the processed count data, over 30GB worth, so we will not do it here. But check out the archs4py Github page and tutorial here: (https://github.com/MaayanLab/archs4py)

Here is some example code to pull data from specific studies from ARCHS4.

```Python
# download archs4 human data
file_path = a4.download.counts("human", path="", version="latest")

#path to file
file = "human_gene_v2.6.h5"

#get sample counts
series_counts = a4.data.series(file, "GSE64016")
```